# TA-005 — Entrenamiento Modelo Ganador: DenseNet-121 con Pesos CheXNet
**Proyecto:** Sistema Multimodal de IA para Apoyo Diagnóstico Clínico Veterinario — Vargas Vet  
**Curso:** Taller Integrador 1 — UPAO  
**Integrantes:** Baylón Toledo, Diogho Matteo (PM) · Saavedra Arroyo, Sebastián Alonso (Scrum Master)  
**Sprint 2:** 31 May – 20 Jun 2026  

**Objetivo del notebook:**
1. Cargar DenseNet-121 con pesos CheXNet (dominio radiológico) en lugar de ImageNet
2. Aplicar mejoras sobre el benchmarking: mayor augmentation, Dropout=0.5, gradient clipping, weight decay=5e-4
3. Entrenar bajo el mismo split y dataset de TA-001/TA-004
4. Alcanzar AUC ≥ 0.80 en validación y F1 ≥ 0.75 en test (umbrales O4 del Project Charter)
5. Exportar modelo final para integración en FastAPI (/predict/radiografia)

## 1 · Verificación de entorno GPU
> Se verifica la disponibilidad de la RTX 4060 antes de iniciar el entrenamiento.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print('Compute capability:', torch.cuda.get_device_properties(0).major, '.', torch.cuda.get_device_properties(0).minor, sep='')
    DEVICE = 'cuda'
else:
    print('ADVERTENCIA: GPU no detectada.')
    DEVICE = 'cpu'

print('Dispositivo activo:', DEVICE)

## 2 · Instalación y verificación de torchxrayvision
> `torchxrayvision` provee pesos CheXNet para DenseNet-121 preentrenado en radiografías torácicas.

In [ ]:
import subprocess, sys

try:
    import torchxrayvision as xrv
    print('torchxrayvision ya instalado:', xrv.__version__)
except ImportError:
    print('Instalando torchxrayvision...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchxrayvision', '-q'])
    import torchxrayvision as xrv
    print('torchxrayvision instalado:', xrv.__version__)

## 3 · Imports y configuración
> Hiperparámetros mejorados respecto al benchmarking (TA-004): Dropout=0.5, weight_decay=5e-4, augmentation más fuerte, gradient clipping.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchxrayvision as xrv
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from pathlib import Path
import time
import json
from tqdm import tqdm
from PIL import Image

sns.set_style('whitegrid')

BASE = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\Radiografias')
MANIFESTS = BASE / 'dataset_split' / 'manifests'
DATASET_SPLIT = BASE / 'dataset_split'
CHECKPOINTS_DIR = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR = Path(r'D:\UPAO_Diogho_Baylon\IX\Taller Integrador I\ModelosIA\modelos\Notebooks')
CHECKPOINTS_DIR.mkdir(exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
CLASSES = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)

EPOCHS_MAX = 30
PHASE1_EPOCHS = 5
BATCH_SIZE = 16
LR = 1e-4
WEIGHT_DECAY = 5e-4
DROPOUT = 0.5
EARLY_STOPPING_PATIENCE = 10
GRAD_CLIP = 1.0

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Configuración TA-005 (mejoras sobre benchmarking):')
print(f'  Device:            {DEVICE}')
print(f'  Pesos iniciales:   CheXNet (torchxrayvision)')
print(f'  Dropout:           {DROPOUT}  (era 0.4 en benchmarking)')
print(f'  Weight decay:      {WEIGHT_DECAY}  (era 1e-4 en benchmarking)')
print(f'  Gradient clipping: {GRAD_CLIP}  (nuevo en TA-005)')
print(f'  Batch size:        {BATCH_SIZE}')
print(f'  Épocas máximas:    {EPOCHS_MAX}')
print(f'  Early stopping:    {EARLY_STOPPING_PATIENCE}')
print(f'  Seed:              {SEED}')

## 4 · Cargar manifests
> Mismo split generado en TA-001 — garantiza comparabilidad directa con el benchmarking.

In [ ]:
df_train = pd.read_csv(MANIFESTS / 'train.csv')
df_val   = pd.read_csv(MANIFESTS / 'val.csv')
df_test  = pd.read_csv(MANIFESTS / 'test.csv')

print('Dataset splits (idéntico a TA-001/TA-004):')
print(f'  Train: {len(df_train)} imágenes | {df_train["PatientName"].nunique()} pacientes')
print(f'  Val:   {len(df_val)} imágenes | {df_val["PatientName"].nunique()} pacientes')
print(f'  Test:  {len(df_test)} imágenes | {df_test["PatientName"].nunique()} pacientes')
print()
print('Distribución de clases en train:')
for cls in CLASSES:
    count = df_train['TAG'].str.contains(cls, na=False).sum()
    print(f'  {cls}: {count} ({count/len(df_train)*100:.1f}%)')

## 5 · Clase VetXRayDataset
> Augmentation mejorado para train: se agregan RandomRotation(10) y ColorJitter para reducir el overfitting observado en TA-004.

In [ ]:
def load_dicom_normalized(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    p2, p98 = np.percentile(img, [2, 98])
    img = np.clip(img, p2, p98)
    img = (img - p2) / (p98 - p2 + 1e-8)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        img = 1.0 - img
    return (img * 255).astype(np.uint8)

def build_label_vector(tag_str, classes):
    vec = np.zeros(len(classes), dtype=np.float32)
    if pd.isna(tag_str):
        vec[-1] = 1.0
        return vec
    tags = [t.strip() for t in str(tag_str).split('|')]
    for i, cls in enumerate(classes):
        if cls in tags:
            vec[i] = 1.0
    if vec.sum() == 0:
        vec[-1] = 1.0
    return vec

class VetXRayDataset(Dataset):
    def __init__(self, df, base_path, classes, split='train'):
        self.df = df.reset_index(drop=True)
        self.base_path = Path(base_path)
        self.classes = classes
        self.split = split
        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dcm_path = self.base_path / self.split / row['FileName']
        img_gray = load_dicom_normalized(dcm_path)
        img_pil = Image.fromarray(img_gray)
        img_rgb = Image.merge('RGB', [img_pil, img_pil, img_pil])
        img_tensor = self.transform(img_rgb)
        label = build_label_vector(row['TAG'], self.classes)
        return img_tensor, torch.from_numpy(label)

print('VetXRayDataset clase definida')
print('Augmentation train: Resize + RandomHorizontalFlip + RandomRotation(10) + ColorJitter')

## 6 · Instanciar datasets y verificar

In [ ]:
train_ds = VetXRayDataset(df_train, DATASET_SPLIT, CLASSES, split='train')
val_ds   = VetXRayDataset(df_val,   DATASET_SPLIT, CLASSES, split='val')
test_ds  = VetXRayDataset(df_test,  DATASET_SPLIT, CLASSES, split='test')

img, lbl = train_ds[0]
print(f'Dataset sizes: train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}')
print(f'Forma tensor: {img.shape}')  # debe ser [3, 224, 224]
print(f'Clases activas en muestra: {[CLASSES[i] for i in range(NUM_CLASSES) if lbl[i]==1]}')

## 7 · Visualizar muestra del dataset con augmentation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (ds, split) in zip(axes, [(train_ds, 'train'), (val_ds, 'val'), (test_ds, 'test')]):
    idx = np.random.randint(0, len(ds))
    img, lbl = ds[idx]
    img_show = img.numpy().transpose(1, 2, 0)
    img_show = (img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    ax.imshow(img_show)
    ax.axis('off')
    tags = [CLASSES[i] for i in range(NUM_CLASSES) if lbl[i] == 1]
    ax.set_title(f'[{split.upper()}]\nTAGs: {" | ".join(tags)}', fontsize=9)

fig.suptitle('Muestra visual con augmentation mejorado — TA-005', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 · Class weights
> Idénticos al benchmarking — misma función de pérdida BCEWithLogitsLoss con pos_weight.

In [ ]:
def compute_class_weights(df, classes):
    N = len(df)
    weights = np.zeros(len(classes))
    for i, cls in enumerate(classes):
        n_pos = df['TAG'].str.contains(cls, na=False).sum()
        weights[i] = (N - n_pos) / n_pos if n_pos > 0 else 1.0
    return weights

class_weights = compute_class_weights(df_train, CLASSES)
class_weights_tensor = torch.from_numpy(class_weights).float().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor)

print('Class weights:')
for cls, w in zip(CLASSES, class_weights):
    print(f'  {cls}: {w:.4f}')

## 9 · Cargar DenseNet-121 con pesos CheXNet
> CheXNet (Rajpurkar et al., 2017) es DenseNet-121 preentrenado en 14 patologías torácicas. Provee features de dominio radiológico superiores a ImageNet para esta tarea.

In [ ]:
chexnet_model = xrv.models.DenseNet(weights='densenet121-res224-chex')

import torch.nn as nn
from torchvision.models import densenet121
from torchvision.models import DenseNet121_Weights

model = densenet121(weights=None)

chexnet_state = chexnet_model.state_dict()
model_state = model.state_dict()

transferred = 0
for key in model_state:
    chex_key = 'densenet121.' + key
    if chex_key in chexnet_state and model_state[key].shape == chexnet_state[chex_key].shape:
        model_state[key] = chexnet_state[chex_key]
        transferred += 1

model.load_state_dict(model_state)

in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(DROPOUT),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'DenseNet-121 con pesos CheXNet cargado:')
print(f'  Capas transferidas desde CheXNet: {transferred}')
print(f'  Parámetros totales: {total_params:,}')
print(f'  In features clasificador: {in_features}')
print(f'  Dropout: {DROPOUT}')

## 10 · Funciones de entrenamiento
> Se agrega gradient clipping (max_norm=1.0) para controlar el overfitting moderado observado en TA-004.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    loss_list = []
    for imgs, lbls in tqdm(loader, desc='Train', leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        loss_list.append(loss.item())
    return np.mean(loss_list)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_list, all_logits, all_labels = [], [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Eval', leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss_list.append(criterion(logits, lbls).item())
            all_logits.append(logits.cpu().numpy())
            all_labels.append(lbls.cpu().numpy())
    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))
    val_auc = roc_auc_score(all_labels, probs, average='macro')
    val_f1  = f1_score(all_labels, (probs > 0.5).astype(int), average='macro', zero_division=0)
    val_acc = accuracy_score(all_labels.flatten(), (probs > 0.5).astype(int).flatten())
    return np.mean(loss_list), val_auc, val_f1, val_acc

print('Funciones definidas (con gradient clipping max_norm=1.0)')

## 11 · Configuración y dataloaders

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Configuración de entrenamiento TA-005:')
print(f'  Batch size:   {BATCH_SIZE}')
print(f'  LR inicial:   {LR}')
print(f'  Weight decay: {WEIGHT_DECAY}')
print(f'  Grad clip:    {GRAD_CLIP}')
print(f'  FASE 1:       {PHASE1_EPOCHS} épocas (feature extraction)')
print(f'  FASE 2:       hasta {EPOCHS_MAX - PHASE1_EPOCHS} épocas (fine-tuning)')
print(f'  Early stop:   patience={EARLY_STOPPING_PATIENCE}')

## 12 · FASE 1 — Feature Extraction (5 épocas)
> Se congela el backbone CheXNet y se entrena solo la cabeza clasificadora para adaptar las salidas a las 5 clases veterinarias.

In [ ]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda')
history_p1 = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': [], 'val_acc': []}

print('\n=== FASE 1: Feature Extraction — CheXNet backbone congelado (5 épocas) ===')
for epoch in range(PHASE1_EPOCHS):
    tl = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    vl, va, vf, vac = eval_epoch(model, val_loader, criterion, DEVICE)
    history_p1['train_loss'].append(tl)
    history_p1['val_loss'].append(vl)
    history_p1['val_auc'].append(va)
    history_p1['val_f1'].append(vf)
    history_p1['val_acc'].append(vac)
    print(f'Época {epoch+1}/{PHASE1_EPOCHS} | TrL: {tl:.4f} | VaL: {vl:.4f} | AUC: {va:.4f} | F1: {vf:.4f} | Acc: {vac:.4f}')

## 13 · FASE 2 — Fine-tuning Completo
> Se descongelan todos los parámetros. ReduceLROnPlateau + early stopping + gradient clipping para controlar el overfitting.

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR/10, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
scaler = torch.amp.GradScaler('cuda')

best_val_auc = 0
best_epoch = 0
ckpt_path = CHECKPOINTS_DIR / 'densenet121_chexnet_best.pth'
history_p2 = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': [], 'val_acc': []}

print('\n=== FASE 2: Fine-tuning Completo ===')
for epoch in range(EPOCHS_MAX - PHASE1_EPOCHS):
    tl = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    vl, va, vf, vac = eval_epoch(model, val_loader, criterion, DEVICE)
    history_p2['train_loss'].append(tl)
    history_p2['val_loss'].append(vl)
    history_p2['val_auc'].append(va)
    history_p2['val_f1'].append(vf)
    history_p2['val_acc'].append(vac)
    if va > best_val_auc:
        best_val_auc = va
        best_epoch = PHASE1_EPOCHS + epoch + 1
        torch.save(model.state_dict(), ckpt_path)
    scheduler.step(va)
    if early_stopping(va):
        print(f'\nEarly stopping en época {PHASE1_EPOCHS + epoch + 1}')
        break
    print(f'Época {PHASE1_EPOCHS + epoch + 1}/{EPOCHS_MAX} | TrL: {tl:.4f} | VaL: {vl:.4f} | AUC: {va:.4f} | F1: {vf:.4f} | Acc: {vac:.4f}')

total_epochs = PHASE1_EPOCHS + len(history_p2['train_loss'])
print(f'\nMejor época: {best_epoch} | AUC val: {best_val_auc:.4f}')
print(f'Cumple O4 (AUC >= 0.80 en val): {best_val_auc >= 0.80}')

## 14 · Curvas de entrenamiento

In [ ]:
all_epochs = list(range(1, total_epochs + 1))
all_train_loss = history_p1['train_loss'] + history_p2['train_loss']
all_val_loss   = history_p1['val_loss']   + history_p2['val_loss']
all_val_auc    = history_p1['val_auc']    + history_p2['val_auc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(all_epochs, all_train_loss, 'o-', label='Train Loss', alpha=0.7)
axes[0].plot(all_epochs, all_val_loss,   's-', label='Val Loss',   alpha=0.7)
axes[0].axvline(x=PHASE1_EPOCHS + 0.5, color='red',    linestyle='--', alpha=0.5, label='Fine-tuning start')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss — DenseNet-121 CheXNet')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(all_epochs, all_val_auc, 'o-', color='green', alpha=0.7, label='Val AUC')
axes[1].axhline(y=0.80, color='green',  linestyle='--', alpha=0.7, label='Umbral O4 (0.80)')
axes[1].axhline(y=0.75, color='orange', linestyle='--', alpha=0.7, label='Umbral O3 (0.75)')
axes[1].axvline(x=best_epoch,           color='red',    linestyle='--', alpha=0.5, label=f'Best ({best_epoch})')
axes[1].axvline(x=PHASE1_EPOCHS + 0.5, color='purple', linestyle='--', alpha=0.5, label='Fine-tuning start')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('AUC (macro)')
axes[1].set_title('AUC validación — DenseNet-121 CheXNet')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle('Curvas de entrenamiento — TA-005 Modelo Ganador', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 15 · Evaluación en Test Set

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()

test_loss, test_auc, test_f1, test_acc = eval_epoch(model, test_loader, criterion, DEVICE)

times = []
with torch.no_grad():
    for i, (imgs, _) in enumerate(test_loader):
        if i >= 2:
            break
        imgs = imgs.to(DEVICE)
        t0 = time.time()
        _ = model(imgs)
        times.append((time.time() - t0) / imgs.size(0))
infer_time = np.mean(times)

print('=== Resultados en TEST SET — TA-005 Modelo Ganador ===')
print(f'  AUC (macro):    {test_auc:.4f} | >= 0.80: {test_auc >= 0.80}')
print(f'  F1-macro:       {test_f1:.4f} | >= 0.75: {test_f1 >= 0.75}')
print(f'  Accuracy:       {test_acc:.4f}')
print(f'  Loss:           {test_loss:.4f}')
print(f'  Inferencia:     {infer_time:.4f} seg/img | < 6 seg: {infer_time < 6}')
print()
print(f'Cumple O4 completo (AUC >= 0.80 Y F1 >= 0.75): {test_auc >= 0.80 and test_f1 >= 0.75}')

## 16 · Comparativa TA-005 vs Benchmarking

In [ ]:
with open(NOTEBOOKS_DIR / 'densenet_results.json') as f:
    r_bench = json.load(f)

print('=== Comparativa DenseNet-121: Benchmarking vs Modelo Final ===')
comparativa = pd.DataFrame({
    'Métrica': ['AUC val', 'AUC test', 'F1-macro test', 'Accuracy test', 'Inferencia (seg)'],
    'TA-004 Benchmarking': [
        f"{r_bench['val_auc']:.4f}",
        f"{r_bench['test_auc']:.4f}",
        f"{r_bench['test_f1_macro']:.4f}",
        f"{r_bench['test_accuracy']:.4f}",
        f"{r_bench['inference_time_sec']:.4f}"
    ],
    'TA-005 Modelo Final': [
        f'{best_val_auc:.4f}',
        f'{test_auc:.4f}',
        f'{test_f1:.4f}',
        f'{test_acc:.4f}',
        f'{infer_time:.4f}'
    ]
})
print(comparativa.to_string(index=False))
print()
mejora_auc = (test_auc - r_bench['test_auc']) * 100
mejora_f1  = (test_f1  - r_bench['test_f1_macro']) * 100
print(f'Mejora AUC test:    {mejora_auc:+.2f} pp')
print(f'Mejora F1-macro:    {mejora_f1:+.2f} pp')

## 17 · Guardar resultados

In [ ]:
results = {
    'model': 'densenet121_chexnet',
    'pretrained_weights': 'CheXNet (densenet121-res224-chex)',
    'val_auc': float(best_val_auc),
    'test_auc': float(test_auc),
    'test_f1_macro': float(test_f1),
    'test_accuracy': float(test_acc),
    'epochs_trained': int(total_epochs),
    'best_epoch': int(best_epoch),
    'inference_time_sec': float(infer_time),
    'batch_size': BATCH_SIZE,
    'dropout': DROPOUT,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip': GRAD_CLIP,
    'cumple_O4_auc': bool(test_auc >= 0.80),
    'cumple_O4_f1': bool(test_f1 >= 0.75),
    'cumple_inferencia': bool(infer_time < 6)
}

results_path = NOTEBOOKS_DIR / 'densenet121_chexnet_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=4)

print(f'Resultados guardados: {results_path}')
print(f'Checkpoint guardado:  {ckpt_path}')

---
## Resumen Final — TA-005

| Criterio O4 | Umbral | Resultado | Estado |
|---|---|---|---|
| AUC en validación | ≥ 0.80 | Ver celda 13 | Ver celda 13 |
| F1-score en test | ≥ 0.75 | Ver celda 15 | Ver celda 15 |
| Inferencia por imagen | < 6 seg | Ver celda 15 | Ver celda 15 |
| Pesos CheXNet aplicados | — | densenet121-res224-chex | ✅ |
| Mejoras anti-overfitting | — | Dropout=0.5, WD=5e-4, GradClip=1.0, Aug+ | ✅ |
| Checkpoint guardado | — | densenet121_chexnet_best.pth | ✅ |

**Próximo paso:** TA-006 — Endpoint FastAPI `/predict/radiografia` integrando este modelo.